In [ ]:
import torch
import numpy as np
from PIL import Image
from copy import deepcopy
from time import time
import os
import pandas as pd
import matplotlib.pyplot as plt
import tqdm
from tqdm import tqdm
import anndata
from HyperST.datasets.hySTDataset import HyperSTGeneADataset
import json

In [ ]:
data_root = "./hest1k_datasets/"  # default data path
dataset = 'colorectum'
data_path = data_root + dataset + "/"
data_process = data_path + 'processed_data/'
ori_st_path = data_path + "st/" # original ST data path
model_name = 'hyperst' # model name
experiment_result_path = "" # experiment path
sample_path = experiment_result_path + "/samples/predict_samples.pt" 
config_path = experiment_result_path + "/config.json"
with open(os.path.join(config_path), "r") as f:
    config_json = json.load(f)
test_slidename_lst = config_json['test_samples']
print("Selected sample list loaded. len of samples: ", len(test_slidename_lst), " list of samples: ", test_slidename_lst)
selected_genes = config_json['selected_genes']
plot_genes = [selected_genes[0], selected_genes[2], selected_genes[4]]
print("Selected gene list loaded. len of selected genes: ", len(selected_genes))
test_dataset = HyperSTGeneADataset(
    slidename_lst=test_slidename_lst,
    selected_genes=selected_genes,
    data_path=data_path,
    process_path=data_process,
    phase='test',
    img_pretrained_model='uni'
)
pred = torch.load(sample_path)
if model_name == 'Stem':
    pred = pred.mean(axis=0)

In [ ]:
idx = 1

In [ ]:
cumlen = np.insert(test_dataset.cumlen, 0, 0) 
sample_name = test_dataset.slidename_lst[idx]
start_id = cumlen[idx]
end_id = cumlen[idx + 1]
origin_data = test_dataset.all_spot_count_mtx_selected_genes[start_id:end_id]
predict_data = pred[start_id:end_id]
# calculate correlation
all_corr = []
for gene_id in range(predict_data.shape[1]):
    x = origin_data[:, gene_id]
    y = predict_data[:, gene_id]
    cor = np.nan_to_num(np.corrcoef(x, y)[0][1])
    all_corr.append(cor)
plt.hist(all_corr, bins=50)
sort_index = np.argsort(all_corr)[::-1]

# Evaluation metrics

In [ ]:
# evaluation metrics
print("sample_name: ", sample_name)
# PCC
print("PCC-10: ", np.mean(sorted(all_corr)[::-1][:10]))
print("PCC-50: ", np.mean(sorted(all_corr)[::-1][:50]))
print("PCC-200: ", np.mean(sorted(all_corr)[::-1][:200]))
# MSE, MAE
print("MSE: ", torch.mean((origin_data - predict_data) ** 2).item())
print("MAE: ", torch.mean(torch.abs(origin_data - predict_data)).item())

# Visualization

In [ ]:
# load test slide
test_adata = anndata.read_h5ad(ori_st_path + sample_name + ".h5ad")
test_adata_filer = test_adata[test_dataset.slice_niche_idx_list[idx]]
origin_data_pd = pd.DataFrame(origin_data.numpy(), columns=selected_genes)
x = test_adata_filer.obsm["spatial"][:, 0]
y = test_adata_filer.obsm["spatial"][:, 1]
print(f'x min {x.min()} max {x.max()} y min {y.min()} max {y.max()}')
x_center = int((x.min() + x.max()) / 2)
y_center = int((y.min() + y.max()) / 2)
patch_size = int(max( (x.max() - x.min() , y.max() - y.min())))
patch_size_overlap = int(patch_size * 0.1)
patch_size = int (patch_size * 1.2)
patch_size_r = int(patch_size / 2)
x_start = x_center - patch_size_r
y_start = y_center - patch_size_r

if y_start < 0:
    y_start = int(y.min() - patch_size_overlap)
print(f'patch_size {patch_size}, patch_size_overlap {patch_size_overlap}, x_start {x_start}, y_start {y_start}')
# plot marker genes

Image.MAX_IMAGE_PIXELS = None

img_path = data_path + "wsis/"
img_raw = Image.open(img_path + sample_name + ".tif") # change suffix according to image type

img_crop = img_raw.crop((x_start, y_start,
                         x_start + patch_size, y_start + patch_size

))
x_crop = test_adata_filer.obsm["spatial"][:, 0] - x_start
y_crop = test_adata_filer.obsm["spatial"][:, 1] - y_start
for gene in plot_genes: # fill in gene names
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))
    gene_idx = np.where(np.array(selected_genes) == gene)[0][0]
    fig.suptitle(f'gene: {gene} PCC: {all_corr[gene_idx]}')
    axs[0].imshow(img_crop)
    color_gt = origin_data_pd.loc[:, origin_data_pd.columns == gene].to_numpy().flatten()
    color_pred = predict_data[:, gene_idx]
    vmin = min(color_gt.min(), color_pred.min())
    vmax = max(color_gt.max(), color_pred.max())
    im0 = axs[0].scatter(x_crop, y_crop, c=color_gt, s=5, alpha=0.9, vmin=vmin, vmax=vmax)
    # axs[0].set_title("ground truth")
    
    axs[1].imshow(img_crop)
    im1 = axs[1].scatter(x_crop, y_crop, c=color_pred, s=5, alpha=0.9, vmin=vmin, vmax=vmax) 
    # axs[1].set_title("pred")

    axs[0].set_xticks([])
    axs[0].set_yticks([])
    axs[1].set_xticks([])
    axs[1].set_yticks([])
    # plt.show()
    fig.colorbar(im0)
    fig.colorbar(im1)
    plt.tight_layout()
    plt.show()